In [ ]:
# Import Libraries

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Deep Learning Libraries
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Input, Concatenate
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Metrics
from sklearn.metrics import classification_report, confusion_matrix

import joblib

# تنظیم Seed برای reproducibility
np.random.seed(42)


In [ ]:
# 1. Generate Synthetic Dataset

n_samples = 5000

data = pd.DataFrame({

    "transaction_amount": np.random.normal(10, 15, n_samples), # مقدار تراکنش

    "transaction_hour": np.random.randint(0, 24, n_samples),

    "merchant_category": np.random.choice(                     # دسته فروشنده
        ["grocery", "electricity", "travel"], n_samples
    ),

    "distance_from_home_km": np.random.uniform(3, 10, n_samples),

    "device_type": np.random.choice(
        ["laptop", "mobile"], n_samples
    ),
# آیا سابقه fraud داشته؟ --> Previous fraud flag
    "previous_flag": np.random.choice(
        ["yes", "no"], n_samples
    ),

    "account_age_years": np.random.randint(3, 12, n_samples)
})


# Targets
# 1. Regression
data["fraud_probability_score"] = np.random.uniform(0, 100, n_samples)

# 2. Binary Classification
data["fraud_label"] = np.where(
    data["fraud_probability_score"] > 60, 1, 0
)
# 3. Multi-class Classification
data["fraud_risk_level"] = pd.cut(
    data["fraud_probability_score"],
    bins=[0, 33, 66, 100],
    labels=["low", "medium", "high"]
)

# pd.cut --> این تابع یک ستون عددی را به بازه‌های دسته‌بندی (bins) تقسیم می‌کند.
# تبدیل عددی به دسته ای multiclass

data.head()



,transaction_amount,transaction_hour,merchant_category,distance_from_home_km,device_type,previous_flag,account_age_years,fraud_probability_score,fraud_label,fraud_risk_level
0,17.450712,7,grocery,7.419728,laptop,yes,6,9.605226,0,low
1,7.926035,0,grocery,4.694693,laptop,yes,11,41.144277,0,medium
2,19.715328,23,grocery,6.028020,laptop,no,11,40.681615,0,medium
3,32.845448,21,electricity,4.787559,laptop,yes,3,88.839442,1,high
4,6.487699,3,grocery,9.190926,mobile,yes,10,21.549345,0,low


In [ ]:
# Introduce Missing Values

# 15% برای transaction_amount و transaction_hour
for col in ["transaction_amount", "transaction_hour"]:
    data.loc[
        data.sample(frac=0.15).index,
        col
    ] = np.nan

# 10% برای distance و device
for col in ["distance_from_home_km", "device_type"]:
    data.loc[
        data.sample(frac=0.10).index,
        col
    ] = np.nan


In [ ]:
# Add Noise

# 5% برای previous_flag
noise_index = data.sample(frac=0.05).index
data.loc[noise_index, "previous_flag"] = np.random.choice(
    ["yes", "no"], len(noise_index)
)

# 20% Noise برای account_age_years
noise_index = data.sample(frac=0.20).index

data.loc[noise_index, "account_age_years"] = data.loc[
    noise_index, "account_age_years"
] + np.random.randint(-3, 4, len(noise_index))


In [ ]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   transaction_amount       4250 non-null   float64 
 1   transaction_hour         4250 non-null   float64 
 2   merchant_category        5000 non-null   str     
 3   distance_from_home_km    4500 non-null   float64 
 4   device_type              4500 non-null   str     
 5   previous_flag            5000 non-null   str     
 6   account_age_years        5000 non-null   int32   
 7   fraud_probability_score  5000 non-null   float64 
 8   fraud_label              5000 non-null   int64   
 9   fraud_risk_level         5000 non-null   category
dtypes: category(1), float64(4), int32(1), int64(1), str(3)
memory usage: 337.2 KB


In [ ]:
# IQR Outlier Handling

def iqr_treatment(df, column):

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[column] = np.where(
        df[column] < lower,
        lower,
        df[column]
    )

    df[column] = np.where(
        df[column] > upper,
        upper,
        df[column]
    )

    return df


In [ ]:
data = iqr_treatment(data, "account_age_years")
data = iqr_treatment(data, "transaction_amount")

In [ ]:
# Split Feature & Target

X = data.drop(
    ["fraud_label", "fraud_risk_level", "fraud_probability_score"],
    axis=1
)

y = data[
    ["fraud_label", "fraud_risk_level", "fraud_probability_score"]
]


In [ ]:
numeric_features = [
    "transaction_amount",
    "transaction_hour",
    "distance_from_home_km",
    "account_age_years"
]

categorical_features = [
    "merchant_category",
    "device_type",
    "previous_flag"
]

feature_names = numeric_features + categorical_features

In [ ]:
# Pipeline + ColumnTransformer

numeric_pipeline = Pipeline([

    ("imputer", SimpleImputer(strategy="median")),

    ("scaler", StandardScaler())

])

categorical_pipeline = Pipeline([

    ("imputer", SimpleImputer(strategy="most_frequent")),

    ("encoder", OneHotEncoder(handle_unknown="ignore"))

])

preprocessor = ColumnTransformer([

    ("num", numeric_pipeline, numeric_features),

    ("cat", categorical_pipeline, categorical_features)

])



In [ ]:
# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y["fraud_label"]
)

In [ ]:
# Final Preprocessed Dataset

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

# Convert DataFrame
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names
)

X_train_processed.head()

,num__transaction_amount,num__transaction_hour,num__distance_from_home_km,num__account_age_years,cat__merchant_category_electricity,cat__merchant_category_grocery,cat__merchant_category_travel,cat__device_type_laptop,cat__device_type_mobile,cat__previous_flag_no,cat__previous_flag_yes
0,-0.994981,0.372692,0.148694,-0.013222,0.0,0.0,1.0,0.0,1.0,1.0,0.0
1,-0.783762,0.059308,0.327350,0.721337,1.0,0.0,0.0,0.0,1.0,0.0,1.0
2,1.176320,-0.724152,1.486463,-0.013222,0.0,1.0,0.0,1.0,0.0,0.0,1.0
3,-1.533198,-0.097384,-1.641576,-1.115061,1.0,0.0,0.0,1.0,0.0,1.0,0.0
4,0.040278,0.059308,1.470246,0.721337,1.0,0.0,0.0,1.0,0.0,0.0,1.0


In [ ]:
y_train = y_train.copy()
y_test = y_test.copy()


risk_encoder = LabelEncoder()

y_train["fraud_risk_level"] = risk_encoder.fit_transform(
    y_train["fraud_risk_level"]
)

y_test["fraud_risk_level"] = risk_encoder.transform(
    y_test["fraud_risk_level"]
)


In [ ]:
# تبدیل Targets

y_train_label = y_train["fraud_label"]
y_test_label = y_test["fraud_label"]

y_train_risk = y_train["fraud_risk_level"]
y_test_risk = y_test["fraud_risk_level"]

y_train_score = y_train["fraud_probability_score"]
y_test_score = y_test["fraud_probability_score"]

In [ ]:
# Model Architecture (Multi‑Task Learning)


input_layer = Input(shape=(X_train_processed.shape[1],), name="input_layer")

# Branch 1 (Binary CLF)
b1 = Dense(64, activation="relu", name="b1_dense1")(input_layer)
b1 = Dense(32, activation="relu", name="b1_dense2")(b1)
b1 = Dense(16, activation="relu", name="b1_dense3")(b1)
branch1_output = Dense(8, activation="relu", name="b1_out")(b1)    # intermediate output


# Branch 2 (Multi-Class Risk CLF)
b2 = Dense(64, activation="relu", name="b2_dense1")(input_layer)
b2 = Dense(32, activation="relu", name="b2_dense2")(b2)
b2 = Dense(16, activation="relu", name="b2_dense3")(b2)
branch2_output = Dense(8, activation="relu", name="b2_out")(b2)   # intermediate output

# Branch 3 - Regression
b3 = Dense(64, activation="relu", name="b3_dense1")(input_layer)
b3 = Dense(32, activation="relu", name="b3_dense2")(b3)
b3 = Dense(16, activation="relu", name="b3_dense3")(b3)
branch3_output = Dense(8, activation="relu", name="b3_out")(b3)   # intermediate output

# Concatenate Branch 1 & 2
concat_12 = Concatenate(name="concat_12")([branch1_output, branch2_output])

# Shared Layers (for classification only)
shared_clf = Dense(32, activation="relu", name="shared_clf_dense1")(concat_12)
shared_clf = Dense(16, activation="relu", name="shared_clf_dense2")(shared_clf)

# Now merge Branch 3 into shared classification layers
concat_all = Concatenate(name="concat_all")([shared_clf, branch3_output])

# Final shared layer before splitting into final outputs
shared_final = Dense(32, activation="relu", name="shared_final")(concat_all)


#ّ Final Outputs

# Binary Classification output
output_label = Dense(1, activation="sigmoid", name="fraud_label")(shared_final)

# Multi-class risk prediction output
output_risk = Dense(3, activation="softmax", name="fraud_risk")(shared_final)

# Regression output
output_score = Dense(1, activation="linear", name="fraud_score")(shared_final)


In [ ]:
# Build Model

model = Model(
    inputs=input_layer,
    outputs=[output_label, output_risk, output_score],
    name="Amirali_Fraud_Detection_ANN"
)

model.summary()

Model: "Amirali_Fraud_Detection_ANN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 11)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ b1_dense1 (Dense)   │ (None, 64)        │        768 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ b2_dense1 (Dense)   │ (None, 64)        │        768 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ b1_dense2 (Dense)   │ (None, 32)        │      2,080 │ b1_dense1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ b2_dense2 (Dense)   │ (None, 32)        │      2,080 │ b2_dense1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ b1_dense3 (Dense)   │ (None, 16)        │        528 │ b1_dense2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ b2_dense3 (Dense)   │ (None, 16)        │        528 │ b2_dense2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ b1_out (Dense)      │ (None, 8)         │        136 │ b1_dense3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ b2_out (Dense)      │ (None, 8)         │        136 │ b2_dense3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ b3_dense1 (Dense)   │ (None, 64)        │        768 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_12           │ (None, 16)        │          0 │ b1_out[0][0],     │
│ (Concatenate)       │                   │            │ b2_out[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ b3_dense2 (Dense)   │ (None, 32)        │      2,080 │ b3_dense1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_clf_dense1   │ (None, 32)        │        544 │ concat_12[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ b3_dense3 (Dense)   │ (None, 16)        │        528 │ b3_dense2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_clf_dense2   │ (None, 16)        │        528 │ shared_clf_dense… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ b3_out (Dense)      │ (None, 8)         │        136 │ b3_dense3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_all          │ (None, 24)        │          0 │ shared_clf_dense… │
│ (Concatenate)       │                   │            │ b3_out[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_final        │ (None, 32)        │        800 │ concat_all[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fraud_label (Dense) │ (None, 1)         │         33 │ shared_final[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fraud_risk (Dense)  │ (None, 3)         │         99 │ shared_final[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fraud_score (Dense) │ (None, 1)         │         33 │ shared_final[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 12,573 (49.11 KB)

 Trainable params: 12,573 (49.11 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Compile Model

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss={
        "fraud_label":"binary_crossentropy",
        "fraud_risk":"sparse_categorical_crossentropy",
        "fraud_score":"mse"
    },
    metrics={
        "fraud_label":"accuracy",
        "fraud_risk":"accuracy",
        "fraud_score":"mae"
    }
)

In [ ]:
# EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [ ]:
# ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=3,
    min_lr=1e-6
)

In [ ]:
# Training

history = model.fit(
    X_train_processed,

    {
        "fraud_label": y_train_label,
        "fraud_risk": y_train_risk,
        "fraud_score": y_train_score,
    },

    validation_split=0.2,
    epochs=20,
    batch_size=32,
    callbacks=[early_stop, reduce_lr]و
    verbose=1
)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (436170912.py, line 15)

In [ ]:
# Evaluation

pred_label, pred_risk, pred_score = model.predict(X_test_processed)

# Binary Classification (Fraud Label)
pred_label_class = (pred_label > 0.5).astype(int)

# Risk Classification
pred_risk_class = pred_risk.argmax(axis=1)


print("Fraud Label Report:")
print(classification_report(y_test_label, pred_label_class))

print("Confusion Matrix:")
print(confusion_matrix(y_test_label, pred_label_class))

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
              precision    recall  f1-score   support

           0       0.60      0.85      0.71       603
           1       0.40      0.16      0.23       397

    accuracy                           0.57      1000
   macro avg       0.50      0.50      0.47      1000
weighted avg       0.52      0.57      0.51      1000

[[511  92]
 [335  62]]


In [ ]:
# Save Model

model.save("model/fraud_ann_model.keras")

# Save Preprocessing

joblib.dump(preprocessor,"model/preprocessor.pkl")
joblib.dump(risk_encoder,"model/risk_encoder.pkl")


['model/risk_encoder.pkl']